###  Regression prediction intervals with MAPIE
**Author: [Carl McBride Ellis](https://www.kaggle.com/carlmcbrideellis)** ([LinkedIn](https://www.linkedin.com/in/carl-mcbride-ellis/))


In this short notebook we calculate [prediction intervals](https://en.wikipedia.org/wiki/Prediction_interval) for the kaggle version of the [California Housing Dataset](https://www.kaggle.com/competitions/playground-series-s3e1) using the [MAPIE - Model Agnostic Prediction Interval Estimator](https://mapie.readthedocs.io/en/latest/index.html) package.

Note that the prediction intervals are not the same as the RMSE of a prediction.
The RMSE is an average of the distances (losses) of the point predictions from the corresponding ground truth values. 
On the other hand the prediction interval represents the range (interval) of values that a particular prediction can reasonably have.
For example, the prediction interval for a house price of 100k represents the range of point predictions our model could make for a house whose true price was actually 
100k;
our model may reasonably predict prices of 90k or 110k, but let us say it is highly unlikely to predict a price of 300k for said house.
Why are prediction intervals important? On kaggle we are very much interested in point predictions, and either minimizing or maximizing the given competition metric. That is all well and good, but what if we were in the market for a house,  our model predicts a price of 100k and said house is on sale for 95k ¿Is that an excellent offer, or is it simply par for the course and we should really be looking for a much cheaper house to best leverage our investment? A point prediction is unable tell us this, meanwhile a prediction interval will indeed guide us as to whether a deal is good or not.

TL;DR: It would be foolhardy to make a business decision based on a point prediction alone that has no associated range. It would be as silly as describing a Normal distribution by its mean ($\mu$), but making no mention of its standard deviation ($\sigma$).

As we expect the house price data to be [heteroscedastic](https://en.wikipedia.org/wiki/Homoscedasticity_and_heteroscedasticity) (*i.e.* here the greater the house price, the greater the variance) we shall use [conformalized quantile regression (CQR)](https://mapie.readthedocs.io/en/latest/examples_regression/4-tutorials/plot_cqr_tutorial.html).

Let us take a quick look at the training data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 20})

train = pd.read_csv("/kaggle/input/playground-series-s3e1/train.csv", index_col="id")
# remove data for houses whose prices were artificially clipped at 500k
train = train.loc[train['MedHouseVal'] < 5.0]
# In the dataset each value corresponds to the average house value in units of 100k
train["MedHouseVal"] = train["MedHouseVal"]*100_000
#take a quick look
train

In [ ]:
X = train
X = X.drop(["MedHouseVal"], axis=1)
y = train["MedHouseVal"]

we shall set aside 1000 rows of data for calibration

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_tmp,  y_train, y_tmp  = train_test_split(X, y,         test_size=2000, random_state=42)
X_calib, X_test, y_calib, y_test = train_test_split(X_tmp, y_tmp, test_size=1000, random_state=42)

print(X_train.shape)
print(X_calib.shape)
print(X_test.shape)

The parameter [`alpha`](https://lightgbm.readthedocs.io/en/latest/Parameters.html#alpha) is 1 - *target coverage*. For example, one would set an `alpha` value of 0.05 in order to obtain a 95% prediction interval. 

Currently accepted base models for the [`MapieQuantileRegressor`](https://mapie.readthedocs.io/en/latest/generated/mapie.regression.MapieQuantileRegressor.html) are:
* scikit-learn [`QuantileRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.QuantileRegressor.html) (based on linear regression)
* scikit-learn [`GradientBoostingRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html)
* scikit-learn [`HistGradientBoostingRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html)
* LightGBM [`LGBMRegressor`](https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMRegressor.html)

It is the latter of these four models that we shall use here:

In [ ]:
alpha = 0.1 # for 90% target coverage

In [ ]:
from lightgbm import LGBMRegressor
regressor = LGBMRegressor( n_estimators       = 1000,
                           learning_rate      = 0.05, 
                           max_depth          = 7, 
                           min_child_samples  = 8,
                           random_state       = 42,
                           objective          = 'quantile',
                           alpha              = alpha
                         )

Install the python [MAPIE](https://github.com/scikit-learn-contrib/MAPIE) package (the only internal dependencies are [scikit-learn](https://scikit-learn.org/stable/index.html) and [numpy](https://numpy.org/)=>1.21)

In [ ]:
!pip install -q mapie 

we shall use the adaptive [conformalized quantile regression (CQR)](https://mapie.readthedocs.io/en/latest/generated/mapie.quantile_regression.MapieQuantileRegressor.html) technique as proposed by [Romano *et al.* (2019)](https://arxiv.org/pdf/1905.03222.pdf) [1,2] via the [`MapieQuantileRegressor`](https://mapie.readthedocs.io/en/latest/generated/mapie.regression.MapieQuantileRegressor.html).
This technique calculates *conformity scores* for each row in the calibration data
<br><br>
$$\large E_i := \max \{l_i - y_{true(i)}, y_{true(i)} - u_i \}$$
<br><br>
then calculates a score, $s$, as the following quantile of the $E_i$ values:
<br><br>
$$\large s = \mathrm{quantile} \left( (1-\alpha)(1+1/n) \right) $$
<br><br>
where $n$ is the number of rows in the calibration set. Finally one subtracts $s$ from the lower quantile for the test set data, and adds $s$ to the upper quantile, thus producing *validity*

In [ ]:
from mapie.regression import MapieQuantileRegressor

mapie = MapieQuantileRegressor(estimator=regressor, cv="split", alpha=alpha)
mapie.fit(X_train, y_train, X_calib=X_calib, y_calib=y_calib)
y_pred, y_pis = mapie.predict(X_test)

let us now save the predictions to a dataframe

In [ ]:
predictions = y_test.to_frame()
predictions.columns = ['y_true']
predictions["point prediction"] = y_pred
predictions["lower"] = y_pis.reshape(-1,2)[:,0]
predictions["upper"] = y_pis.reshape(-1,2)[:,1]
# take a quick look
predictions

Finally let us now plot the predictions and their corresponding prediction intervals:

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))

plt.errorbar(predictions["y_true"], predictions["point prediction"], 
             yerr=(predictions["point prediction"]-predictions["lower"], predictions["upper"]-predictions["point prediction"]),
             ecolor='grey', linestyle='', marker = "o", capsize=5)

ax.axline([0, 0], [1, 1], color = "red", linestyle='--', lw=3, zorder=3)
plt.xlim(0)
plt.ylim(0)
plt.xlabel('True house price ($)')
plt.ylabel('Predicted house price ($)')
plt.show();

### Error plot (sorted by prediction interval width)

In [ ]:
predictions["error"] = predictions["point prediction"] - predictions["y_true"]

predictions["error_upper"] =   (predictions["upper"] - predictions["point prediction"])
predictions["error_lower"] =  -(predictions["point prediction"] - predictions["lower"])

# sort by total interval width
predictions["interval_width"] = predictions["upper"] - predictions["lower"]
sorted_predictions = predictions.sort_values(by=['interval_width']).reset_index(drop=True)

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8))

plt.plot(sorted_predictions["error"], 'o', markersize=3, label="error (y_pred - y_true)")

plt.fill_between(np.arange(len(sorted_predictions)),
                 sorted_predictions["error_lower"], 
                 sorted_predictions["error_upper"], 
                 alpha=0.5, color="grey", label="prediction interval")

ax.axline([0, 0], [1, 0], color = "red", linestyle='--', lw=2, zorder=3, label="y_true")
plt.xticks([])
plt.xlim([0, len(sorted_predictions)])
plt.ylabel("Errors")
plt.legend(loc="upper left", fontsize=14)
plt.show()

Let us calculate the % of point predictions that were situated within the prediction interval

In [ ]:
# count number of points outside of predicted interval
sorted_predictions["is_outside_range"] = 0
sorted_predictions["is_outside_range"] = sorted_predictions["is_outside_range"].where((
    (sorted_predictions["error"] < sorted_predictions["error_upper"]) & (sorted_predictions["error"] > sorted_predictions["error_lower"]) ), 
    other=1)

print(round(100-(100/len(sorted_predictions))*sorted_predictions["is_outside_range"].sum(),1))

That is all well and good. However, perhaps more importantly, let us now calculate the % of prediction intervals that contained the actual ground truth value (the marginal coverage)

In [ ]:
# count number of prediction intervals that actually contain the ground truth value
sorted_predictions["gt_within_PI"] = 0
sorted_predictions["gt_within_PI"] = sorted_predictions["gt_within_PI"].where((
    (sorted_predictions["y_true"] < sorted_predictions["upper"]) & (sorted_predictions["y_true"] > sorted_predictions["lower"]) ), 
    other=1)

print(round(100-(100/len(sorted_predictions))*sorted_predictions["gt_within_PI"].sum(),1))

In [ ]:
# re-sort for plot
sorted_predictions = predictions.sort_values(by=['y_true']).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(30, 9))

plt.plot(sorted_predictions["y_true"], 'o', markersize=3, label="y_true")

plt.fill_between(np.arange(len(sorted_predictions)),
                 sorted_predictions["lower"], 
                 sorted_predictions["upper"], 
                 alpha=0.5, color="grey", label="prediction interval")

plt.xticks([])
plt.xlim([0, len(sorted_predictions)])
plt.ylabel("True value")
plt.legend(loc="upper left", fontsize=14)
plt.show()

Average prediction interval width

In [ ]:
print(round(predictions["interval_width"].mean(),-2))
print(round(predictions["interval_width"].median(),-2))

### Model performance evaluation: The Winkler Interval score
We shall now assess the overall performance of our model by calculating the mean of the [Winkler Interval score](https://otexts.com/fpp3/distaccuracy.html#winkler-score) [3,4,5] which, for an individual interval, is given by:

$$\large W_{\alpha} = \begin{cases}
    (u-l)  +  \frac{2}{\alpha}(l-y),  & \mbox{if  } y < l \\
    (u-l)               ,            & \mbox{if  } l \le y \le u \\
    (u-l)  +  \frac{2}{\alpha}(y-u) , & \mbox{if  } y > u
\end{cases} $$

where $y$ is the true value, $u$ it the upper prediction interval, $l$ is the lower prediction interval, and $\alpha$ is (1-coverage). Note that the Winkler Interval score constitutes a proper scoring rule [4,5].

In [ ]:
# for local evaluation add the dataset https://www.kaggle.com/datasets/carlmcbrideellis/winkler-interval-score-metric
import sys
sys.path.append('/kaggle/input/winkler-interval-score-metric/')
import MWIS_metric
# help(MWIS_metric.score)

MWIS,coverage = MWIS_metric.score(predictions["y_true"],predictions["lower"],predictions["upper"],alpha)
print(f"MWI score          ",round(MWIS,3))
print("Predictions coverage   ", round(coverage*100,1),"%")

### Related reading

**Documentation**
* [MAPIE - Model Agnostic Prediction Interval Estimator](https://mapie.readthedocs.io/en/latest/index.html)
* [Tutorial for conformalized quantile regression (CQR)](https://mapie.readthedocs.io/en/latest/examples_regression/4-tutorials/plot_cqr_tutorial.html) - MAPIE documentation
* [MAPIE (GitHub)](https://github.com/scikit-learn-contrib/MAPIE)

**Papers**
* [Vianney Taquet, Vincent Blot, Thomas Morzadec, Louis Lacombe, and Nicolas Brunel "*MAPIE: an open-source library for distribution-free uncertainty quantification*", arXiv:2207.12274 (2022)](https://arxiv.org/pdf/2207.12274v1.pdf)
* [Thibault Cordier, Vincent Blot, Louis Lacombe, Thomas Morzadec, Arnaud Capitaine, Nicolas Brunel "*Flexible and Systematic Uncertainty Estimation with Conformal Prediction via the MAPIE library*", Proceedings of Machine Learning Research ***204*** pp. 549-581 (2023)](https://proceedings.mlr.press/v204/cordier23a/cordier23a.pdf)
* [1] [Yaniv Romano, Evan Patterson, and Emmanuel J. Candès. “*Conformalized Quantile Regression*”, Advances in neural information processing systems **32** (2019)](https://arxiv.org/pdf/1905.03222.pdf)
* [2] [Matteo Sesia, and Emmanuel J. Candès "*A comparison of some conformal quantile regression methods*", Stat **9** e261 (2020)](https://doi.org/10.1002/sta4.261)
* [Rina Foygel Barber, Emmanuel J. Candès, Aaditya Ramdas, and Ryan J. Tibshirani "*Predictive inference with the jackknife+*", Annals of Statistics **49** pp. 486-507 (2021)](https://www.doi.org/10.1214/20-AOS1965)
* [3] [Robert L. Winkler "*A Decision-Theoretic Approach to Interval Estimation*", Journal of the American Statistical Association, **67**, pp. 187-191 (1972)](https://doi.org/10.1080/01621459.1972.10481224)
* [4] [Tilmann Gneiting and Adrian E Raftery "*Strictly Proper Scoring Rules, Prediction, and Estimation*", Journal of the American Statistical Association, **102**, pp. 359-378 (2007)](https://doi.org/10.1198/016214506000001437) (Section 6.2)
* [5]  [Jonas R. Brehmer, and Tilmann Gneiting "*Scoring interval forecasts: Equal-tailed, shortest, and modal interval*", Bernoulli **volume 27** pp. 1993-2010 (2021)](https://doi.org/10.3150/20-BEJ1298)
* [Valeriy Manokhin "*Practical Guide to Applied Conformal Prediction*", Packt Publishing (2023)](https://www.packtpub.com/product/practical-guide-to-applied-conformal-prediction/9781805122760) (Chapter 7)
* [Anastasios N. Angelopoulos, Stephen Bates "*A Gentle Introduction to Conformal Prediction and Distribution-Free Uncertainty Quantification*", arXiv:2107.07511 (2022)](https://arxiv.org/pdf/2107.07511.pdf) (§ 2.2)

**Blogs** *etc.*
* ["With MAPIE, uncertainties are back in machine learning"](https://towardsdatascience.com/with-mapie-uncertainties-are-back-in-machine-learning-882d5c17fdc3) by Vianney Taquet
* [“MAPIE” Explained Exactly How You Wished Someone Explained to You](https://towardsdatascience.com/mapie-explained-exactly-how-you-wished-someone-explained-to-you-78fb8ce81ff3) by Samuele Mazzanti
* ["How to Predict Risk-Proportional Intervals with Conformal Quantile Regression"](https://towardsdatascience.com/how-to-predict-risk-proportional-intervals-with-conformal-quantile-regression-175775840dc4) by Samuele Mazzanti
* [Awesome Conformal Prediction](https://github.com/valeman/awesome-conformal-prediction) - a large collection of conformal prediction resources collated by Valeriy Manokhin
* ["How to predict quantiles in a more intelligent way (or ‘Bye-bye quantile regression, hello Conformalized Quantile Regression’)"](https://valeman.medium.com/how-to-predict-quantiles-in-a-more-intelligent-way-or-bye-bye-quantile-regression-hello-24a65e4c50f) by Valeriy Manokhin
* [Prediction intervals explained: A LightGBM tutorial](https://developer.ibm.com/articles/prediction-intervals-explained-a-lightgbm-tutorial/) - IBM

**Notebooks**
* [Prediction intervals: Quantile Regression Forests](https://www.kaggle.com/code/carlmcbrideellis/prediction-intervals-quantile-regression-forests) by Carl McBride Ellis